# Nvidia Nemotron Reasoning Challenge

## Score: .54

In [ ]:
import subprocess, sys, os
from pathlib import Path
def resolve_python_path(target_dir):
    for pth_file in Path(target_dir).glob("*.pth"):
        with pth_file.open() as fp:
            relpath = fp.read()
            rel_pack_path = (pth_file.parent/relpath)
            if rel_pack_path.exists():
                print(f"append {rel_pack_path}")
                sys.path.append(str(rel_pack_path))


offline_dir = "/kaggle/input/nvidia-nemotron-offline-packages/offline_packages"
target_dir = "/kaggle/working/packages"

os.makedirs(target_dir, exist_ok=True)
resolve_python_path("/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/")

if os.path.exists(offline_dir):
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--no-index",
        "--find-links", offline_dir,
        "--target", target_dir,
        "--no-deps",
        "datasets", "trl"
    ])
    print("Installed from offline packages")

sys.path.append(target_dir)
resolve_python_path(target_dir)

import datasets, cutlass

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import sys, stat, shutil, gc, zipfile
import polars as pl
import torch
import torch.nn.functional as F
import kagglehub
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected. In Kaggle, set Accelerator to GPU before running training.")
print(f"torch={torch.__version__} | cuda={torch.cuda.is_available()} | gpu={torch.cuda.get_device_name(0)}")

import importlib.util
import types
from pathlib import Path as _Path

_BUNDLED_NEMOTRON_EVAL_UTILS = r'''
"""
Shared helpers for Nemotron competition: training text, inference prompt parity, boxed extraction,
and offline error bucketing. Keep in sync with Kaggle vLLM eval (chat template + enable_thinking=False).

Copy this file next to notebooks on Kaggle (e.g. upload to /kaggle/working/) so imports resolve.
"""

from __future__ import annotations

import math
import re
from typing import Any

# Must match what you use in training / what you expect at inference.
CHAT_TEMPLATE_KWARGS: dict[str, Any] = {"enable_thinking": False}

# Canonical user suffix (track 3: same string at train and eval).
USER_PROMPT_SUFFIX = "\nReturn your final answer in \\boxed{}."


def user_content_from_raw_prompt(raw_prompt: str) -> str:
    p = (raw_prompt or "").strip()
    return p + USER_PROMPT_SUFFIX


def build_messages_training(
    raw_prompt: str,
    answer: str,
    cot: str,
) -> list[dict[str, str]]:
    user_msg = user_content_from_raw_prompt(raw_prompt)
    assistant_msg = f"{cot}\n\n\\boxed{{{answer}}}"
    return [
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": assistant_msg},
    ]


def format_sft_text(
    tokenizer,
    raw_prompt: str,
    answer: str,
    cot: str,
    *,
    tokenize: bool = False,
    add_generation_prompt: bool = False,
) -> str:
    messages = build_messages_training(raw_prompt, answer, cot)
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=tokenize,
            add_generation_prompt=add_generation_prompt,
            chat_template_kwargs=CHAT_TEMPLATE_KWARGS,
        )
    except Exception:
        user_msg = user_content_from_raw_prompt(raw_prompt)
        assistant_msg = f"{cot}\n\n\\boxed{{{answer}}}"
        return (
            f"<|im_start|>user\n{user_msg}<|im_end|>\n"
            f"<|im_start|>assistant\n{assistant_msg}<|im_end|>"
        )


def build_messages_inference(raw_prompt: str) -> list[dict[str, str]]:
    return [{"role": "user", "content": user_content_from_raw_prompt(raw_prompt)}]


def apply_inference_prompt(tokenizer, raw_prompt: str) -> str:
    messages = build_messages_inference(raw_prompt)
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        chat_template_kwargs=CHAT_TEMPLATE_KWARGS,
    )


def task_family(prompt: str) -> str:
    """Heuristic labels from eda.ipynb — for stratified sampling (track 1)."""
    s = str(prompt)[:1200].lower()
    if "8-bit binary" in s or ("bit manipulation" in s and "binary" in s):
        return "bit_8"
    if "secret encryption" in s or ("decrypt" in s and "wonderland" in s):
        return "text_crypto"
    if "numeral system" in s:
        return "numeral_convert"
    if "unit conversion" in s:
        return "unit_conv"
    if "gravitational" in s or "falling distance" in s:
        return "physics"
    if "equation" in s and "transformation" in s:
        return "symbol_equation"
    return "unclassified"


def extract_boxed(text: str) -> str | None:
    """Last complete \\boxed{...} with balanced braces."""
    if not text:
        return None
    best: str | None = None
    i = 0
    needle = r"\boxed{"
    while True:
        j = text.find(needle, i)
        if j < 0:
            break
        start = j + len(needle)
        depth = 1
        k = start
        while k < len(text) and depth:
            c = text[k]
            if c == "{":
                depth += 1
            elif c == "}":
                depth -= 1
            k += 1
        if depth == 0:
            best = text[start : k - 1].strip()
        i = start
    return best


def _try_float(s: str) -> float | None:
    s = s.strip()
    if not s:
        return None
    try:
        return float(s)
    except ValueError:
        m = re.match(r"^[-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?$", s)
        if m:
            try:
                return float(m.group(0))
            except ValueError:
                return None
    return None


def answers_match(pred: str, gold: str, *, rel_tol: float = 1e-5, abs_tol: float = 1e-8) -> bool:
    """Loose match: exact string after strip, or numeric closeness."""
    p, g = pred.strip(), gold.strip()
    if p == g:
        return True
    fp, fg = _try_float(p), _try_float(g)
    if fp is not None and fg is not None:
        return math.isclose(fp, fg, rel_tol=rel_tol, abs_tol=abs_tol)
    return False


def stratified_sample_indices(prompts: list[str], n_target: int, seed: int) -> list[int]:
    """
    Roughly balance heuristic task families (track 1). If a family is too small, backfill from the rest.
    """
    import numpy as np
    import pandas as pd

    n = len(prompts)
    if n_target >= n:
        return list(range(n))
    df = pd.DataFrame({"idx": range(n), "family": [task_family(p) for p in prompts]})
    rng = np.random.default_rng(seed)
    families = df["family"].unique().tolist()
    rng.shuffle(families)
    k = len(families)
    base = n_target // k
    extra = n_target % k
    picked: set[int] = set()
    for j, fam in enumerate(families):
        want = base + (1 if j < extra else 0)
        sub = df[df["family"] == fam]["idx"].tolist()
        rng.shuffle(sub)
        for idx in sub[:want]:
            picked.add(int(idx))
    shortfall = n_target - len(picked)
    if shortfall > 0:
        rest = [i for i in range(n) if i not in picked]
        rng.shuffle(rest)
        picked.update(int(x) for x in rest[:shortfall])
    out = list(picked)[:n_target]
    rng.shuffle(out)
    return out


def error_bucket(
    full_completion: str,
    gold_answer: str,
    *,
    extracted: str | None = None,
) -> str:
    """
    Track 5: coarse buckets for targeting data / prompts.
    - no_boxed: model did not emit extractable \\boxed{}
    - wrong_answer: boxed present but does not match gold
    - correct: match
    """
    ext = extracted if extracted is not None else extract_boxed(full_completion)
    if ext is None:
        return "no_boxed"
    if answers_match(ext, gold_answer):
        return "correct"
    return "wrong_answer"

'''

def _load_nemotron_eval_utils():
    for p in (
        _Path("/kaggle/working/nemotron_eval_utils.py"),
        _Path.cwd() / "nemotron_eval_utils.py",
    ):
        if p.is_file():
            spec = importlib.util.spec_from_file_location("nemotron_eval_utils", p)
            mod = importlib.util.module_from_spec(spec)
            assert spec.loader is not None
            spec.loader.exec_module(mod)
            return mod
    mod = types.ModuleType("nemotron_eval_utils")
    exec(
        compile(_BUNDLED_NEMOTRON_EVAL_UTILS, "nemotron_eval_utils.py", "exec"),
        mod.__dict__,
    )
    return mod


neu = _load_nemotron_eval_utils()


In [ ]:
def _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,
                     group_size=None, norm_before_gate=True, upcast=True):
    dtype = x.dtype
    if upcast: x = x.float()
    variance = x.pow(2).mean(-1, keepdim=True)
    x_normed = x * torch.rsqrt(variance + eps)
    out = x_normed * weight.float()
    if bias is not None: out = out + bias.float()
    if z is not None: out = out * F.silu(z.float())
    return out.to(dtype)

for name, mod in list(sys.modules.items()):
    if hasattr(mod, 'rmsnorm_fn'):
        mod.rmsnorm_fn = _pure_rmsnorm_fn

src = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell"
dst = "/tmp/ptxas-blackwell"
if os.path.exists(src):
    shutil.copy2(src, dst)
    os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    import triton.backends.nvidia as nv_backend
    src_bin = os.path.join(os.path.dirname(nv_backend.__file__), "bin")
    dst_bin = "/tmp/triton_nvidia_bin"
    shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
    for f in os.listdir(dst_bin):
        fp = os.path.join(dst_bin, f)
        if os.path.isfile(fp):
            os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    nv_backend.__file__ = os.path.join(dst_bin, "..", "__init__.py")
    os.environ["TRITON_PTXAS_PATH"] = dst
    print("Triton ptxas fix applied.")

In [ ]:
LORA_RANK = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05
MAX_SEQ_LEN = 2048
NUM_EPOCHS = 1
BATCH_SIZE = 1
GRAD_ACCUM = 4
LR = 5e-5
WARMUP_RATIO = 0.03
MAX_COT_CHARS = 10000
MAX_TRAIN_ROWS = 4000
MIX_OFFICIAL_TRAIN_CSV = False
STRATIFIED_FAMILY_BALANCE = True
STRATIFIED_SEED = 42
PLACEHOLDER_COT = "Work step by step from the prompt to the boxed answer."
OFFICIAL_TRAIN_PATHS = [
    "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv",
    "/kaggle/input/nvidia-nemotron-model-reasoning-challenge/train.csv",
]

RUN_NAME = f"run24_push_a{LORA_ALPHA}_e{NUM_EPOCHS}_lr{str(LR).replace('.', 'p')}_n{MAX_TRAIN_ROWS}_mix{int(MIX_OFFICIAL_TRAIN_CSV)}_strat{int(STRATIFIED_FAMILY_BALANCE)}"
OUTPUT_DIR = f"/kaggle/working/adapter_{RUN_NAME}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
train_df = pl.read_csv("/kaggle/input/datasets/kienngx/nemotron-30b-competition-trainingdata-cot-labels/final_Nemotron_training_data.csv")
n_raw = len(train_df)
train_df = train_df.with_columns(
    pl.col("prompt").cast(pl.Utf8, strict=False).fill_null("").str.strip_chars(),
    pl.col("answer").cast(pl.Utf8, strict=False).fill_null("").str.strip_chars(),
    pl.col("generated_cot").cast(pl.Utf8, strict=False).fill_null("").str.strip_chars(),
)
train_df = train_df.filter(
    pl.col("prompt").str.len_chars() > 0,
    pl.col("answer").str.len_chars() > 0,
    pl.col("generated_cot").str.len_chars() > 0,
)
train_df = train_df.unique(subset=["prompt"], keep="first")
train_df = train_df.with_columns(
    pl.when(pl.col("generated_cot").str.len_chars() > MAX_COT_CHARS)
    .then(pl.col("generated_cot").str.slice(0, MAX_COT_CHARS) + "...")
    .otherwise(pl.col("generated_cot"))
    .alias("generated_cot"),
)
print(f"clean cot: {n_raw} -> {len(train_df)} rows")

if MIX_OFFICIAL_TRAIN_CSV:
    for _path in OFFICIAL_TRAIN_PATHS:
        if os.path.exists(_path):
            official = pl.read_csv(_path)
            if {"prompt", "answer"}.issubset(official.columns):
                official = official.select(
                    pl.col("prompt").cast(pl.Utf8, strict=False).fill_null("").str.strip_chars(),
                    pl.col("answer").cast(pl.Utf8, strict=False).fill_null("").str.strip_chars(),
                ).with_columns(pl.lit(PLACEHOLDER_COT).alias("generated_cot"))
                official = official.filter(
                    pl.col("prompt").str.len_chars() > 0,
                    pl.col("answer").str.len_chars() > 0,
                )
                train_df = pl.concat([train_df, official], how="vertical")
                print(f"mixed official train.csv: +{len(official)} rows from {_path}")
            break

train_df = train_df.unique(subset=["prompt"], keep="first")
n_pool = len(train_df)
if n_pool > MAX_TRAIN_ROWS:
    if STRATIFIED_FAMILY_BALANCE:
        _idx = neu.stratified_sample_indices(train_df["prompt"].to_list(), MAX_TRAIN_ROWS, STRATIFIED_SEED)
        _pdf = train_df.to_pandas()
        train_df = pl.from_pandas(_pdf.iloc[_idx].reset_index(drop=True))
    else:
        train_df = train_df.sample(n=MAX_TRAIN_ROWS, seed=42)

hf_dataset = Dataset.from_pandas(train_df.to_pandas())
print(f"Run: {RUN_NAME} | train rows: {len(hf_dataset)} (pool {n_pool}) | output: {OUTPUT_DIR}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def build_training_text(example):
    text = neu.format_sft_text(
        tokenizer,
        example["prompt"],
        example["answer"],
        example["generated_cot"],
    )
    return {"text": text}

hf_dataset = hf_dataset.map(build_training_text, remove_columns=hf_dataset.column_names)



In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map={"": 0},
    trust_remote_code=True,
    dtype=torch.bfloat16,
)
model.gradient_checkpointing_enable()

for name, mod in sys.modules.items():
    if "modeling_nemotron_h" in name:
        mod.is_fast_path_available = False
        print(f"Patched {name}: is_fast_path_available = False")

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules="all-linear",
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

import triton.backends.nvidia.compiler as nv_compiler
os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = "/tmp/ptxas-blackwell"
nv_compiler.get_ptxas_version = lambda arch: "12.0"

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    logging_steps=5,
    bf16=True,
    max_grad_norm=1.0,
    optim="adamw_torch",
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    save_strategy="no",
    report_to="none",
    dataset_text_field="text",
    max_length=MAX_SEQ_LEN,
    packing=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

trainer = SFTTrainer(
    model=model,
    train_dataset=hf_dataset,
    processing_class=tokenizer,
    args=training_args,
)

print("Starting training...")
trainer.train()
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Adapter saved to {OUTPUT_DIR}:")
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  {f} ({size/1024:.1f} KB)")

In [ ]:
import zipfile
import os

zip_path = "/kaggle/working/submission.zip"

print(f"Packaging files from {OUTPUT_DIR}...")


with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(OUTPUT_DIR):
        fpath = os.path.join(OUTPUT_DIR, fname)
        zf.write(fpath, fname) 

print(f"Created {zip_path} ({os.path.getsize(zip_path) / 1024 / 1024:.1f} MB)")

with zipfile.ZipFile(zip_path, 'r') as zf:
    zip_contents = zf.namelist()
    print(f"Zip Contents: {zip_contents}")
    
    if "adapter_config.json" not in zip_contents:
        raise AssertionError("CRITICAL ERROR: adapter_config.json is missing from the zip. The Kaggle evaluation will fail.")
    if "adapter_model.safetensors" not in zip_contents:
         raise AssertionError("CRITICAL ERROR: adapter_model.safetensors is missing from the zip.")

print("submission.zip is ready")